In [ ]:
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader

from src.utils.datasets import CUB200Dataset, extract_embeddings

# Fix random seed to achieve reproducibility of the results
torch.manual_seed(42)
device = (
    torch.accelerator.current_accelerator()
    if torch.accelerator.is_available()
    else torch.device("cpu")
)
device

### CUB200-2011 dataset


In [ ]:
# Download a pretrained ResNet50 model and remove the classifier (last fully
# connected layer) to generate image embeddings for the CUB200 dataset
resnet50 = torchvision.models.resnet50(pretrained=True)
resnet50.fc = nn.Identity()

In [3]:
# These transformations preprocess the input images to match the format and statistics
# expected by models pre-trained on ImageNet, such as the ResNet-50
transform = torchvision.transforms.Compose(
    [
        torchvision.transforms.Resize((224, 224)),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
        ),
    ]
)

# The dataset is downloaded if neccessary
trainset = CUB200Dataset(train=True, transform=transform, download=False)
testset = CUB200Dataset(train=False, transform=transform, download=False)
trainloader = DataLoader(trainset, batch_size=32, num_workers=4)
testloader = DataLoader(testset, batch_size=32, num_workers=4)

In [ ]:
# Extract output of the ResNet-50's last hidden layer (embeddings) and save them
extract_embeddings(trainloader, resnet50, "cub200_train_embed.pt", device)
extract_embeddings(testloader, resnet50, "cub200_test_embed.pt", device)